# Task 2 — Missing-value imputation & category cleaning

**Problem-statement step:** *There are missing income values for some customers. Conduct missing value imputation, considering that customers with similar education and marital status tend to have comparable yearly incomes, on average. ... scrutinize the categories of education and marital status for data cleaning.*

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

# Locate the raw data whether the notebook is opened from its own folder
# (notebooks/) or from the project root.
import os
_CANDIDATES = ["marketing_data.csv", "../marketing_data.csv",
               os.path.join(os.path.dirname(os.getcwd()), "marketing_data.csv")]
DATA_PATH = next((p for p in _CANDIDATES if os.path.exists(p)), "marketing_data.csv")
print("Using data file:", DATA_PATH)

Using data file: ../marketing_data.csv


## 1. Load & fix dtypes (from Task 1)

In [2]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
df["Income"] = (df["Income"].astype("string")
                .str.replace(r"[\$,]", "", regex=True).str.strip().astype(float))
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%m/%d/%y")

## 2. Scrutinise the categorical variables

In [3]:
df["Education"].value_counts()

Education
Graduation    1127
PhD            486
Master         370
2n Cycle       203
Basic           54
Name: count, dtype: int64

In [4]:
df["Marital_Status"].value_counts()

Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
YOLO          2
Absurd        2
Name: count, dtype: int64

`Marital_Status` contains junk / data-entry values: **`Alone`, `YOLO`, `Absurd`**. `Alone` clearly means single; the 4 junk rows are recoded to `Single` as the most plausible category.

In [5]:
df["Marital_Status"] = df["Marital_Status"].replace(
    {"Alone": "Single", "YOLO": "Single", "Absurd": "Single"})
df["Marital_Status"].value_counts()

Marital_Status
Married     864
Together    580
Single      487
Divorced    232
Widow        77
Name: count, dtype: int64

## 3. Impute missing income by (Education, Marital_Status) peers
Income is right-skewed with an extreme outlier (666,666), so we fill each gap with the **group median** rather than the mean.

In [6]:
print("Missing before:", int(df["Income"].isna().sum()))
grp_median = df.groupby(["Education", "Marital_Status"])["Income"].median().round(0)
grp_median.head(10)

Missing before: 24


Education  Marital_Status
2n Cycle   Divorced          49118.0
           Married           46462.0
           Single            48668.0
           Together          45774.0
           Widow             47682.0
Basic      Divorced           9548.0
           Married           22352.0
           Single            16383.0
           Together          23179.0
           Widow             22123.0
Name: Income, dtype: float64

In [7]:
df["Income"] = df.groupby(["Education", "Marital_Status"])["Income"].transform(
    lambda s: s.fillna(s.median()))
df["Income"] = df["Income"].fillna(df["Income"].median())
print("Missing after :", int(df["Income"].isna().sum()))
df["Income"].describe().round(2)

Missing after : 0


count      2240.00
mean      52230.72
std       25039.99
min        1730.00
25%       35538.75
50%       51222.50
75%       68289.75
max      666666.00
Name: Income, dtype: float64

### Conclusion
* `Marital_Status` cleaned from 8 raw labels → 5 meaningful categories.
* All 24 missing incomes imputed from education + marital-status peer medians.